<a href="https://colab.research.google.com/github/rubyratcha-19/ge338_lab2/blob/main/Lab2_6606614805.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Lab 2: Spatial Analysis กับ Google Earth Engine**

6606614805 รัชชานนท์ สุรินทร์

In [10]:
# ===========================================
# Imports + Initialize
# ===========================================
import ee
ee.Authenticate()
ee.Initialize(project='inlaid-reactor-457503-u6')

# ===========================================
# Province Boundary (Nan)
# ===========================================
nan = (ee.FeatureCollection("FAO/GAUL/2015/level1")
       .filter(ee.Filter.And(
           ee.Filter.eq('ADM0_NAME','Thailand'),
           ee.Filter.eq('ADM1_NAME','Nan')
       )))

# ===========================================
# Cloud Mask Function
# ===========================================
def maskS2(img):
    scl = img.select('SCL')
    mask = scl.neq(3)\
        .And(scl.neq(7))\
        .And(scl.neq(8))\
        .And(scl.neq(9))\
        .And(scl.neq(10))\
        .And(scl.neq(11))
    return img.updateMask(mask).divide(10000)

# ===========================================
# Image Collections
# ===========================================
before = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(nan)
          .filterDate('2023-02-01','2023-03-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',40))
          .map(maskS2))

after = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(nan)
         .filterDate('2023-05-01','2023-08-31')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',40))
         .map(maskS2))

before_img = before.median().clip(nan)
after_img  = after.median().clip(nan)

# ===========================================
# Spectral Indices (NDVI + NDWI + NDBI)
# ===========================================
# NDVI: vegetation
ndvi_before = before_img.normalizedDifference(['B8','B4']).rename('NDVI')
ndvi_after  = after_img.normalizedDifference(['B8','B4']).rename('NDVI')
ndvi_diff   = ndvi_after.subtract(ndvi_before).rename('NDVI_Diff')

# NDWI: water
ndwi_before = before_img.normalizedDifference(['B3','B8']).rename('NDWI')
ndwi_after  = after_img.normalizedDifference(['B3','B8']).rename('NDWI')
ndwi_diff   = ndwi_after.subtract(ndwi_before).rename('NDWI_Diff')

# NDBI: built-up
ndbi_before = before_img.normalizedDifference(['B11','B8']).rename('NDBI')
ndbi_after  = after_img.normalizedDifference(['B11','B8']).rename('NDBI')
ndbi_diff   = ndbi_after.subtract(ndbi_before).rename('NDBI_Diff')

# ===========================================
# Visualization Parameters
# ===========================================
rgb = {'bands':['B4','B3','B2'], 'min':0, 'max':0.3}
ndvi_vis = {'min':0, 'max':1, 'palette':['yellow','green']}
ndwi_vis = {'min':0, 'max':1, 'palette':['blue','cyan']}
ndbi_vis = {'min':-0.5,'max':0.5,'palette':['white','red']}
diff_vis = {'min':-0.5,'max':0.5,'palette':['red','yellow','green']}

# ===========================================
# Create Map
# ===========================================
m = geemap.Map()
m.centerObject(nan, 8)

# RGB
m.addLayer(before_img.visualize(**rgb), {}, 'RGB Before')
m.addLayer(after_img.visualize(**rgb), {}, 'RGB After')

# NDVI
m.addLayer(ndvi_before, ndvi_vis, 'NDVI Before')
m.addLayer(ndvi_after, ndvi_vis, 'NDVI After')
m.addLayer(ndvi_diff, diff_vis, 'NDVI Diff')

# NDWI
m.addLayer(ndwi_before, ndwi_vis, 'NDWI Before')
m.addLayer(ndwi_after, ndwi_vis, 'NDWI After')
m.addLayer(ndwi_diff, diff_vis, 'NDWI Diff')

# NDBI
m.addLayer(ndbi_before, ndbi_vis, 'NDBI Before')
m.addLayer(ndbi_after, ndbi_vis, 'NDBI After')
m.addLayer(ndbi_diff, diff_vis, 'NDBI Diff')

# Boundary
m.addLayer(nan.style(color='black', fillColor='00000000'), {}, 'Nan Province')

m

Map(center=[18.847712206673826, 100.83316508361675], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# ===========================================
# Export ONLY Diff Layers (GeoTIFF)
# ===========================================

export_diff = {
    'NDVI_Diff': ndvi_diff,
    'NDWI_Diff': ndwi_diff,
    'NDBI_Diff': ndbi_diff
}

for name, img in export_diff.items():
    task = ee.batch.Export.image.toDrive(
        image=img,  # ❗ ไม่ต้อง visualize → จะได้ค่า real
        description=name,
        folder='GEE_Nan_Diff_Export',
        fileNamePrefix=name.lower(),
        region=nan.geometry(),
        scale=10,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()

print("🚀 Export Diff (.tif) started!")

In [ ]:
# ===========================================
# Export Nan Boundary (Shapefile)
# ===========================================

task = ee.batch.Export.table.toDrive(
    collection=nan,
    description='Nan_Boundary',
    folder='Nan_Project',   # โฟลเดอร์ใน Google Drive
    fileNamePrefix='Nan_boundary',
    fileFormat='SHP'
)

task.start()